In [13]:
!pip install -q --upgrade bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.3 MB/s eta 0:00:00


In [1]:
import torch
import subprocess

print("=== TORCH GPU INFO ===")
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
    print("CUDA Version:", torch.version.cuda)
    print("Current Device:", torch.cuda.current_device())
    print("Device Capability:", torch.cuda.get_device_capability(0))
else:
    print("No GPU detected")

print("\n=== NVIDIA-SMI ===")
try:
    print(subprocess.check_output(["nvidia-smi"]).decode())
except:
    print("nvidia-smi not available")

=== TORCH GPU INFO ===
GPU Available: True
GPU Name: Tesla T4
GPU Count: 1
CUDA Version: 12.8
Current Device: 0
Device Capability: (7, 5)

=== NVIDIA-SMI ===
Thu Apr 16 16:31:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       3MiB /  15360MiB |      

In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [4]:
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [5]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

In [6]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)


In [9]:
tokenizer = AutoTokenizer.from_pretrained(PHI)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    PHI,
    device_map="auto",
    quantization_config=quant_config
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [12]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Tell a joke for a room of Data ScientistsSure, here's a joke tailored for data scientists:

Why did the data scientist break up with the computer?

Because it kept asking for more space!


In [14]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=5120, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_feature

In [7]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch

def generate(model_name, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    streamer = TextStreamer(tokenizer)

    if quant:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        streamer=streamer
    )
    return outputs


In [9]:
generate(QWEN, messages)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

<|im_start|>user
Tell a joke for a room of Data Scientists<|im_end|>
<|im_start|>assistant
Absolutely! Here's a *data scientist-approved* joke with a dash of Python, regression, and a sprinkle of humor:

---

Why did the data scientist break up with the statistician?

Because of the *correlation* — they were always *highly* predictive, but never *causal*.

And the real reason? They kept saying, *"It's just a regression model — not


tensor([[151644,    872,    198,  40451,    264,  21646,    369,    264,   3054,
            315,   2885,  56016, 151645,    198, 151644,  77091,    198,  77045,
              0,   5692,    594,    264,    353,    691,  27468,  66262,      9,
          21646,    448,    264,  23911,    315,  13027,     11,  30549,     11,
            323,    264,  84847,    315,  27385,   1447,  44364,  10234,   1521,
            279,    821,  27468,   1438,    705,    448,    279,  42489,   1103,
           1939,  17949,    315,    279,    353,   6005,  22221,      9,   1959,
            807,   1033,   2677,    353,  11892,    398,      9,  59236,     11,
            714,   2581,    353,    924,  35414,      9,    382,   3036,    279,
           1931,   2874,     30,   2379,   8604,   5488,     11,  91204,   2132,
            594,   1101,    264,  30549,   1614,   1959,    537]],
       device='cuda:0')

In [11]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=200)

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<｜begin▁of▁sentence｜><｜User｜>Tell a joke for a room of Data Scientists<｜Assistant｜><think>
Okay, so I need to come up with a joke for a room full of data scientists. Hmm, data scientists are all about numbers, stats, and maybe some quirky stuff. Let me think about how they would joke about that.

First, maybe something about their job. They collect data, analyze it, so maybe a joke that involves numbers or statistics. Or perhaps something more about their tools or methods. Wait, the example given was about the number of letters in their names. That's a good start because it's relatable and ties into their field.

So, the joke was: "Why did the data scientist go to the doctor? To get the data on the patient!" That's a classic setup. Data scientists are the ones who collect and analyze data, so they would want to check the data's quality, which is why they go to the doctor for that check.

Let me think of another angle. Maybe something about their workflow or how they work. For example, 

tensor([[151646, 151644,  40451,    264,  21646,    369,    264,   3054,    315,
           2885,  56016, 151645, 151648,    198,  32313,     11,    773,    358,
           1184,    311,   2525,    705,    448,    264,  21646,    369,    264,
           3054,   2480,    315,    821,  13923,     13,  88190,     11,    821,
          13923,    525,    678,    911,   5109,     11,  10472,     11,    323,
           7196,   1045,  67132,   6259,     13,   6771,    752,   1744,    911,
           1246,    807,   1035,  21646,    911,    429,    382,   5338,     11,
           7196,   2494,    911,    862,   2618,     13,   2379,   6530,    821,
             11,  23643,    432,     11,    773,   7196,    264,  21646,    429,
          17601,   5109,    476,  13142,     13,   2521,   8365,   2494,    803,
            911,    862,   7375,    476,   5413,     13,  13824,     11,    279,
           3110,   2661,    572,    911,    279,   1372,    315,  11931,    304,
            862,   5036,    